## Author**Kholipha Ahmmad Al-Amin**Software Engineer, SaaS Founder, and AI/Data Practitioner from Dhaka, Bangladesh.Portfolio: https://kholipha-ahmmad-al-amin.equisaas-bd.comGitHub: https://github.com/kholipha-ahmmad-al-aminLinkedIn: https://www.linkedin.com/in/kholipha-ahmmad-al-amin

# Early Detection Dengue Fever based on CBC data using machine learning

This notebook implements a machine learning pipeline for early detection of Dengue fever using Complete Blood Count (CBC) data. We explore multiple models, evaluate their performance, provide explainability via SHAP, and deploy a simple Gradio interface for predictions.

## Overview
- **Dataset**: CBC features like Haemoglobin, WBC, Platelets, etc., with binary labels (0: Negative, 1: Positive for Dengue).
- **Models**: CatBoost, RandomForest, HistGradientBoosting, SVM.
- **Evaluation**: Cross-validation, accuracy, F1-score, confusion matrices, ROC curves.
- **Explainability**: SHAP for feature importance.
- **Deployment**: Gradio app for interactive predictions.

Run cells sequentially. Ensure your dataset (`dengue_dataset.csv`) is uploaded to Colab.

#Setup Environment

In [ ]:
# Step 1: Install Required Packages
# We install packages quietly (-q flag) to keep the output clean.
# Key libraries: CatBoost for gradient boosting, scikit-learn for ML basics,
# SHAP for explainability, Gradio for the UI, and joblib for model persistence.
!pip install -q catboost scikit-learn shap gradio joblib

# Step 2: Import Essential Libraries
# Data handling: pandas, numpy
# Visualization: matplotlib, seaborn
# ML: train_test_split, cross_val_score, metrics, scalers, and classifiers
# Other: joblib for saving models, shap for explanations, Google Drive for storage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
import joblib
import shap
from google.colab import drive
import os

## Data Loading and Preprocessing

Here, we mount Google Drive to access persistent storage, copy the dataset if needed, load it into a Pandas DataFrame, handle missing values by imputing medians, and prepare features (X) and target (y).

In [ ]:
# Mount Google Drive for persistent storage
# This allows us to save models, scalers, and plots without losing them between sessions.
drive.mount('/content/drive')

# Define base path in Drive for organizing files
base_path = '/content/drive/MyDrive/Mosiur_Rahaman_Dengue_Project/'
os.makedirs(base_path, exist_ok=True)  # Create directory if it doesn't exist
dataset_path = os.path.join(base_path, 'dengue_dataset.csv')

# Copy the uploaded dataset to Drive (assuming it's initially in /content/)
import shutil
shutil.copy('/content/dengue_dataset.csv', dataset_path)
print(f"Dataset saved to {dataset_path}")

# Load the CSV dataset into a DataFrame
df = pd.read_csv(dataset_path)

# Basic Preprocessing:
# - Gender and Result are already encoded as 0/1 (binary).
# - Fill any missing numeric values with column medians to avoid data loss.
df.fillna(df.median(numeric_only=True), inplace=True)

# Separate features (X) and target (y: Dengue Result)
X = df.drop('Result', axis=1)
y = df['Result']

# Optional: Quick peek at the data
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

## Model Training and Evaluation

We scale the features (essential for SVM), split the data into train/test sets with stratification to maintain class balance, define four classifiers, perform cross-validation, train on the full train set, evaluate on test, and save models/plots.

Metrics include accuracy and macro F1 (balanced for imbalanced classes). We also generate confusion matrices, ROC curves (for probabilistic models), and feature importance plots (for tree-based models).

In [ ]:
# Scale features using StandardScaler
# Normalization is crucial for distance-based models like SVM.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save the fitted scaler for later use in predictions
joblib.dump(scaler, os.path.join(base_path, 'scaler.pkl'))

# Split data: 80% train, 20% test, stratified to preserve class distribution, fixed random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Define a dictionary of models to compare:
# - CatBoost: Gradient boosting with categorical support (verbose=False to suppress logs)
# - RandomForest: Ensemble of decision trees
# - HistGB: Histogram-based gradient boosting (faster alternative)
# - SVM: Support Vector Classifier with RBF kernel for non-linear boundaries
models = {
    'CatBoost': CatBoostClassifier(random_state=42, verbose=False),
    'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1),  # Use all CPU cores
    'HistGB': HistGradientBoostingClassifier(random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)  # probability=True for ROC curves
}

# Initialize results dictionary and 5-fold stratified cross-validation
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loop over each model for training and evaluation
for name, model in models.items():
    print(f"Training {name}...")  # Progress indicator

    # Perform 5-fold CV for robust estimates
    acc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro')

    # Fit the model on the entire training set
    model.fit(X_train, y_train)

    # Generate predictions on test set
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    # Calculate key metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    report = classification_report(y_test, y_pred)
    print(report)  # Print detailed classification report

    # Store results including the trained model
    results[name] = {'Accuracy': acc, 'Macro F1': f1, 'CV Accuracy': acc_scores.mean(), 'CV F1': f1_scores.mean(), 'Model': model}

    # Save the trained model to Drive
    joblib.dump(model, os.path.join(base_path, f'{name}_model.pkl'))

    # Plot 1: Confusion Matrix (visualize true vs. predicted labels)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(base_path, f'{name}_cm.png'), dpi=1200, bbox_inches='tight')
    plt.close()

    # Plot 2: ROC Curve (only for models with probabilities)
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'{name} ROC Curve')
        plt.legend(loc="lower right")
        plt.savefig(os.path.join(base_path, f'{name}_roc.png'), dpi=1200, bbox_inches='tight')
        plt.close()

    # Plot 3: Feature Importance (only for tree-based models)
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        feat_imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values('Importance', ascending=True)
        plt.figure(figsize=(10, 6))
        sns.barplot(data=feat_imp_df, x='Importance', y='Feature')
        plt.title(f'{name} Feature Importance')
        plt.savefig(os.path.join(base_path, f'{name}_feature_imp.png'), dpi=1200, bbox_inches='tight')
        plt.close()

# Compile and display model comparison table
comparison_df = pd.DataFrame(results).T[['Accuracy', 'Macro F1', 'CV Accuracy', 'CV F1']]
print("\nModel Comparison Summary:")
print(comparison_df.round(4))

# Identify and announce the best model based on Macro F1 score
best_name = comparison_df['Macro F1'].idxmax()
best_model = results[best_name]['Model']
print(f"\nBest performing model: {best_name} (Macro F1: {comparison_df.loc[best_name, 'Macro F1']:.4f})")

## Explainability with SHAP

Using SHAP (SHapley Additive exPlanations), we generate a summary plot for the best model to understand feature contributions to predictions. We also create a comparison table image and a radar (spider) chart for model metrics.

In [ ]:
# Initialize SHAP Explainer for the best model
# Uses the training data as background for accurate explanations.
explainer = shap.Explainer(best_model, X_train)
shap_values = explainer(X_test)

# Generate SHAP Summary Plot: Shows global feature impact across test samples
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=False)
plt.savefig(os.path.join(base_path, 'shap_summary.png'), dpi=1200, bbox_inches='tight')
plt.close()
print("SHAP summary plot saved.")

# Save Model Comparison Table as Image
# Useful for reports or presentations.
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=comparison_df.round(4).values,
                 colLabels=comparison_df.columns,
                 rowLabels=comparison_df.index,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
plt.savefig(os.path.join(base_path, 'model_comparison_table.png'), dpi=1200, bbox_inches='tight')
plt.close()

# Radar Chart for Multi-Metric Comparison
# Visualizes how models perform across metrics in a polar plot.
metrics = ['Accuracy', 'Macro F1', 'CV Accuracy', 'CV F1']
num_vars = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for name in models.keys():
    values = [comparison_df.loc[name, metric] for metric in metrics]
    values += values[:1]  # Close the polygon
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=name)
    ax.fill(angles, values, alpha=0.1)
ax.set_ylim(0, 1)  # Scale from 0 to 1
ax.set_yticklabels([])  # Hide radial ticks
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
plt.title('Model Performance Radar Chart', size=16, y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.savefig(os.path.join(base_path, 'model_spider_chart.png'), dpi=1200, bbox_inches='tight')
plt.close()
print("Comparison visuals saved.")

## Interactive Deployment with Gradio

This section creates a user-friendly web interface for inputting CBC values and getting predictions with SHAP explanations. It can be run independently after training (mount Drive and ensure files are saved).

**Note**: In Colab, this launches a public shareable link. For production, consider Hugging Face Spaces.

In [ ]:
!pip install -q gradio shap catboost

import gradio as gr
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from google.colab import drive
import os
import tempfile
import warnings

# Suppress Matplotlib warnings for a cleaner output
warnings.filterwarnings("ignore", category=UserWarning, module='matplotlib')

# --- 1. SETUP AND MODEL LOADING ---
# Mount Google Drive (re-authenticate if needed)
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Google Drive mount failed: {e}. Running with local file paths (ensure files are present).")

# Define paths to saved artifacts
# Use a try-except block to handle environments where Drive might not be available
try:
    base_path = '/content/drive/MyDrive/Mosiur_Rahaman_Dengue_Project/'
    if not os.path.exists(base_path):
        raise FileNotFoundError
except (FileNotFoundError, NameError):
    print("Google Drive path not found. Defaulting to current directory './'.")
    base_path = './'
    # Create dummy files if they don't exist, to prevent crashes.
    # In a real scenario, you would upload these files to your Colab/local environment.
    if not os.path.exists('scaler.pkl'):
        print("WARNING: 'scaler.pkl' not found. The app will likely fail.")
    if not os.path.exists('CatBoost_model.pkl'):
        print("WARNING: 'CatBoost_model.pkl' not found. The app will likely fail.")
    if not os.path.exists('dengue_dataset.csv'):
        print("WARNING: 'dengue_dataset.csv' not found. SHAP background data will be limited.")


scaler_path = os.path.join(base_path, 'scaler.pkl')
best_model_path = os.path.join(base_path, 'CatBoost_model.pkl')
dataset_path = os.path.join(base_path, 'dengue_dataset.csv')

# Load the scaler and best model
try:
    scaler = joblib.load(scaler_path)
    best_model = joblib.load(best_model_path)
except FileNotFoundError as e:
    print(f"Fatal Error: Could not load model or scaler from path. {e}")
    # Exit if essential files are missing
    exit()

# Define feature names (must match dataset columns)
features = ['Gender', 'Age', 'Haemoglobin', 'ESR', 'WBC', 'Neutrophil', 'Lymphocyte', 'Monocyte', 'Eosinophil', 'Basophil', 'RBC', 'Platelets']

# --- 2. SHAP EXPLAINER INITIALIZATION ---
# Prepare SHAP background data (sample from training for efficiency)
try:
    # Sample from the full dataset for a robust background
    full_df = pd.read_csv(dataset_path)
    # Ensure the columns in the sample match the 'features' list order
    df_sample = full_df[features].sample(n=min(100, len(full_df)), random_state=42)
    X_sample_scaled = scaler.transform(df_sample)
except (FileNotFoundError, KeyError) as e:
    print(f"Warning: Could not load dataset for SHAP background. Using random data. {e}")
    # Fallback to random data if the dataset is not available
    X_sample_scaled = np.random.rand(100, len(features))

# Initialize SHAP Explainer
explainer = shap.Explainer(best_model, X_sample_scaled)

# --- 3. CORE PREDICTION & EXPLANATION FUNCTION ---
def predict_dengue(gender, age, haemoglobin, esr, wbc, neutrophil, lymphocyte, monocyte, eosinophil, basophil, rbc, platelets):
    # Bundle inputs into a DataFrame with correct feature order
    input_data = pd.DataFrame([[gender, age, haemoglobin, esr, wbc, neutrophil, lymphocyte, monocyte, eosinophil, basophil, rbc, platelets]],
                              columns=features)

    # Scale the input for the model
    input_scaled = scaler.transform(input_data)

    # Get prediction probability
    prob = best_model.predict_proba(input_scaled)[0][1]

    # Create the dictionary for the gr.Label component
    # This automatically handles coloring and labeling
    prediction_label = {
        "Dengue Positive": prob,
        "Dengue Negative": 1 - prob
    }

    # Format the probability text for the textbox
    probability_text = f"The model predicts a {prob:.2%} probability of the patient being Dengue Positive."

    # Generate SHAP explanation for the specific prediction
    shap_values_instance = explainer(input_scaled)

    # Create SHAP force plot and save it to a temporary file
    # This ensures thread-safety and avoids overwriting plots
    fig = shap.force_plot(explainer.expected_value, shap_values_instance.values[0], input_data, matplotlib=True, show=False, text_rotation=15)

    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
        fig.savefig(tmpfile.name, dpi=150, bbox_inches='tight')
        plt.close(fig) # Important: close the plot to free memory
        plot_path = tmpfile.name

    return prediction_label, probability_text, plot_path

# --- 4. STUNNING UI/UX with GRADIO BLOCKS AND CSS ---

# Custom CSS for a premium, stylish look
custom_css = """
body { background-color: #f0f2f6; } /* Light gray background for better contrast */
.gradio-container {
    background: linear-gradient(135deg, #1e293b 0%, #334155 100%);
    border-radius: 20px !important;
    padding: 2rem !important;
}
.app-title {
    text-align: center;
    font-size: 3rem;
    font-weight: 900;
    color: #ffffff;
    margin-bottom: 0.5rem;
    letter-spacing: -2px;
    text-shadow: 0 4px 10px rgba(0,0,0,0.3);
}
.app-subtitle {
    text-align: center;
    font-size: 1.2rem;
    color: #cbd5e1;
    margin-bottom: 2rem;
}
.section-header { color: #e2e8f0; border-bottom: 2px solid #475569; padding-bottom: 5px; margin-bottom: 1rem; }
.input-group, .output-group {
    background-color: rgba(51, 65, 85, 0.7); /* Semi-transparent dark slate */
    border: 1px solid #475569;
    border-radius: 16px;
    padding: 1.5rem;
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37);
    backdrop-filter: blur(4px); /* Glassmorphism effect */
}
#prediction-label .label-name { font-size: 1.5rem !important; font-weight: bold !important; }
#prediction-label .confidence { font-size: 1.2rem !important; }
#shap-plot { border: 1px solid #475569; border-radius: 12px; overflow: hidden; }
#analyze-btn { font-size: 1.2rem !important; font-weight: bold !important; }
footer { display: none !important } /* Hides default Gradio footer */
"""

with gr.Blocks() as demo:
    # --- Header ---
    gr.Markdown(
        """
        <div class='app-title'>Dengue Guard AI</div>
        <div class='app-subtitle'>Advanced Early Detection of Dengue Using CBC Data</div>
        """
    )

    with gr.Row(equal_height=False):
        # --- INPUTS COLUMN ---
        with gr.Column(scale=1):
            with gr.Group(elem_classes="input-group"):
                gr.Markdown("<h3 class='section-header'>👤 Patient Information</h3>",)
                age = gr.Slider(minimum=1, maximum=100, value=30, step=1, label="Age (Years)")
                gender = gr.Radio([0, 1], label="Gender (0=Female, 1=Male)", value=1)

            with gr.Group(elem_classes="input-group"):
                gr.Markdown("<h3 class='section-header'>🩸 Complete Blood Count (CBC)</h3>")
                with gr.Row():
                    haemoglobin = gr.Slider(minimum=5.0, maximum=20.0, value=13.0, label="Haemoglobin (g/dL)")
                    platelets = gr.Slider(minimum=10, maximum=500, value=150, label="Platelets (x10^9/L)")
                with gr.Row():
                    wbc = gr.Slider(minimum=1.0, maximum=30.0, value=7.0, label="WBC (x10^9/L)")
                    rbc = gr.Slider(minimum=2.0, maximum=7.0, value=4.5, label="RBC (x10^12/L)")
                with gr.Row():
                    neutrophil = gr.Slider(minimum=10, maximum=90, value=60, label="Neutrophil (%)")
                    lymphocyte = gr.Slider(minimum=5, maximum=70, value=30, label="Lymphocyte (%)")
                with gr.Accordion("Show Advanced Parameters", open=False):
                    esr = gr.Slider(minimum=0, maximum=150, value=10, label="ESR (mm/hr)")
                    monocyte = gr.Slider(minimum=1, maximum=15, value=5, label="Monocyte (%)")
                    eosinophil = gr.Slider(minimum=0, maximum=10, value=2, label="Eosinophil (%)")
                    basophil = gr.Slider(minimum=0, maximum=5, value=0, label="Basophil (%)")

            # Group all input components for the function call
            inputs = [gender, age, haemoglobin, esr, wbc, neutrophil, lymphocyte, monocyte, eosinophil, basophil, rbc, platelets]

            analyze_btn = gr.Button("Analyze Patient Data", variant="primary", elem_id="analyze-btn")

        # --- OUTPUTS COLUMN ---
        with gr.Column(scale=2):
            with gr.Group(elem_classes="output-group"):
                gr.Markdown("<h2 class='section-header'>🔬 Analysis Results</h2>")
                pred_label = gr.Label(label="Prediction", elem_id="prediction-label")
                prob_text = gr.Textbox(label="Prediction Confidence", interactive=False)

            with gr.Group(elem_classes="output-group"):
                gr.Markdown("<h3 class='section-header'>📊 Feature Impact Analysis (SHAP)</h3>")
                gr.Markdown("The plot below shows which features pushed the prediction towards 'Dengue Positive' (red) or 'Dengue Negative' (blue). Longer bars have a greater impact.")
                shap_plot = gr.Image(label="SHAP Force Plot", interactive=False, elem_id="shap-plot")

    # --- Examples Section ---
    gr.Markdown("<h3 class='section-header' style='text-align:center; margin-top: 2rem;'>Sample Patient Profiles</h3>")
    gr.Examples(
        examples=[
            [1, 25, 14.0, 20, 4.0, 75, 15, 8, 1, 0, 5.0, 60],   # Classic Dengue: Low Platelets, Low WBC, High Neutrophil
            [0, 30, 13.0, 10, 7.0, 60, 30, 5, 2, 0, 4.5, 250], # Healthy Profile
            [1, 50, 11.0, 40, 12.0, 70, 20, 8, 1, 0, 4.0, 50],  # Strong Dengue Indicators
            [0, 20, 13.5, 5, 5.5, 45, 45, 8, 1, 1, 4.8, 180]    # Borderline/Viral symptoms
        ],
        inputs=inputs,
        outputs=[pred_label, prob_text, shap_plot],
        fn=predict_dengue,
        cache_examples=True # Caches results for faster example loading
    )

    # --- Footer ---
    gr.Markdown(
        """
        <div style='text-align: center; color: #94a3b8; margin-top: 2rem;'>
        <strong>Disclaimer:</strong> This tool is an AI demonstration and not a substitute for professional medical advice.
        Predictions are based on a machine learning model and should be verified by a qualified healthcare provider.
        </div>
        """
    )

    # Connect the button to the function
    analyze_btn.click(
        fn=predict_dengue,
        inputs=inputs,
        outputs=[pred_label, prob_text, shap_plot]
    )

# Launch the Gradio interface
demo.launch(debug=True, css=custom_css)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.0/139.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.5 MB/s eta 0:00:00
Mounted at /content/drive
Caching examples at: '/content